# 数据类型与类型转换

学习目标：根据数值范围与存储需求选择 dtype，完成类型转换，并识别溢出和类型提升的边界。

前置知识：Python 数值类型、数组创建与形状、基本算术。

运行环境：Python 3.12、NumPy 2.5。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例使用单元内构造的数据，后续单元沿用首次导入的 np。

标明“预期异常”的单元会直接显示原始报错；阅读异常类型与原因后，继续运行下一单元。

## 1 指定与查看 dtype

三个传感器各记录一个整数读数，计算半个单位的校正值前，可以先把读数转换为浮点数组。

dtype 描述数组元素的数据类型。创建时用 dtype 参数指定；已有数组用 astype() 转换，默认返回同形的新数组，原数组不变。下面使用 int16 保存原始整数，用 float32 进行计算。

In [1]:
import numpy as np

readings = np.array([18, 20, 22], dtype=np.int16)
floating = readings.astype(np.float32)
corrected = floating + 0.5

print(readings, readings.dtype)  # [18 20 22] int16，原始数组不变。
print(corrected, corrected.dtype)  # [18.5 20.5 22.5] float32。
print(floating.shape, corrected.shape)  # 均为 (3,)，转换不改变形状。

[18 20 22] int16
[18.5 20.5 22.5] float32
(3,) (3,)


## 2 常用数值类型

类型名中的数字表示一个元素的位数，例如 int16 占 16 位。布尔、有符号整数、无符号整数、浮点数和复数适合不同的数据。

| 类型 | 中文名称／含义 | 示例用途 |
| --- | --- | --- |
| np.bool_ | 布尔值，True 或 False | 条件是否成立 |
| np.int16 | 16 位有符号整数 | 可正可负的整数读数 |
| np.uint8 | 8 位无符号整数 | 0～255 的非负整数 |
| np.float32 | 32 位浮点数 | 有限精度的小数计算 |
| np.float64 | 64 位浮点数 | 比 float32 更高的表示精度 |
| np.complex128 | 128 位复数，实部与虚部各用 float64 | 含实部与虚部的数值 |

这些是常用例子，不是全部类型。明确要求位宽时使用带位数的名称；默认整数类型的位宽受平台影响。

In [2]:
flags = np.array([True, False], dtype=np.bool_)
offsets = np.array([-2, 3], dtype=np.int16)
levels = np.array([0, 255], dtype=np.uint8)
fractions = np.array([0.25, 0.5], dtype=np.float64)
signals = np.array([1 + 2j, 3 - 1j], dtype=np.complex128)

print(flags, flags.dtype)  # [ True False] bool。
print(offsets, offsets.dtype)  # [-2  3] int16。
print(levels, levels.dtype)  # [  0 255] uint8。
print(fractions, fractions.dtype)  # [0.25 0.5 ] float64。
print(signals, signals.dtype)  # [1.+2.j 3.-1.j] complex128。

[ True False] bool
[-2  3] int16
[  0 255] uint8
[0.25 0.5 ] float64
[1.+2.j 3.-1.j] complex128


## 3 类型推断与数组标量

### 3.1 从输入推断类型

省略 dtype 时，np.array() 会根据全部输入推断共同类型。混合整数与浮点数的列表可以得到浮点数组，不能只看第一个元素判断类型。

In [3]:
integers = np.array([2, 4, 6])
mixed = np.array([2, 4.5, 6])
with_complex = np.array([2.0, 1 + 2j])

print(integers.dtype)  # 本机为 int64；默认整数位宽需以实际平台为准。
print(mixed, mixed.dtype)  # [2.  4.5 6. ] float64，统一为浮点类型。
print(with_complex, with_complex.dtype)  # [2.+0.j 1.+2.j] complex128。

int64
[2.  4.5 6. ] float64
[2.+0.j 1.+2.j] complex128


### 3.2 数组标量与 Python 标量

从普通数值数组中取一个元素，通常得到带 dtype 的 NumPy 数组标量（array scalar），不是 Python 内置标量。需要 Python int 时，可以显式调用 int()。

下面 readings[0] 取第一个元素。数组标量和含一个值的零维 ndarray 是不同对象，后者的形状为 ()。

In [4]:
readings = np.array([18, 20], dtype=np.int16)
array_scalar = readings[0]
python_scalar = int(array_scalar)
zero_dimensional = np.array(18, dtype=np.int16)

print(type(array_scalar), array_scalar.dtype)  # numpy.int16，dtype 为 int16。
print(type(python_scalar), python_scalar)  # Python int，值为 18。
print(type(zero_dimensional), zero_dimensional.shape)  # numpy.ndarray，形状 ()。

<class 'numpy.int16'> int16
<class 'int'> 18
<class 'numpy.ndarray'> ()


## 4 元素大小与存储量

itemsize 是一个元素的字节数；nbytes 是全部元素的字节数，等于 size × itemsize。对于相同形状的数组，dtype 会影响元素存储量。

nbytes 不包含数组对象的其他属性开销，也不是整个进程的内存占用。下面两个数组均有三行两列，行表示观测，列表示传感器。

In [5]:
compact = np.array([[10, 20], [11, 21], [12, 22]], dtype=np.int16)
wide = compact.astype(np.float64)

print(compact.shape, compact.itemsize, compact.nbytes)  # (3, 2) 2 12。
print(wide.shape, wide.itemsize, wide.nbytes)  # (3, 2) 8 48。
print(compact.size * compact.itemsize == compact.nbytes)  # True。

(3, 2) 2 12
(3, 2) 8 48
True


## 5 查询类型范围与精度

### 5.1 iinfo：整数范围

np.iinfo() 返回指定整数类型的信息，min 和 max 是可表示的最小值与最大值。选择整数 dtype 时，既要容纳输入，也要考虑后续运算结果。

In [6]:
signed = np.iinfo(np.int8)
unsigned = np.iinfo(np.uint8)
wider = np.iinfo(np.int16)

print(signed.min, signed.max)  # -128 127。
print(unsigned.min, unsigned.max)  # 0 255，无符号整数不能表示负数。
print(wider.min, wider.max)  # -32768 32767。

-128 127
0 255
-32768 32767


### 5.2 finfo：浮点范围与间距

np.finfo() 查询浮点类型。min 和 max 是最小与最大的有限数值；eps 是 1.0 与比它大的下一个可表示浮点数之间的差。

eps 描述 1.0 附近的间距，不是任意计算的误差上限。float64 的 eps 比 float32 小，但两者都只有有限精度。

In [7]:
single = np.finfo(np.float32)
double = np.finfo(np.float64)

print(single.min, single.max)  # 约 -3.40e38、3.40e38。
print(double.min, double.max)  # 约 -1.80e308、1.80e308。
print(single.eps, double.eps)  # 约 1.19e-7、2.22e-16。

-3.4028235e+38 3.4028235e+38
-1.7976931348623157e+308 1.7976931348623157e+308
1.1920929e-07 2.220446049250313e-16


## 6 转换与运算的边界

### 6.1 窄化转换

从较宽的类型转换到较窄的类型，称为窄化转换。例如 int16 转为 int8 会减少每个元素的存储量，但超出目标范围的值会改变。

astype() 默认允许有损转换。浮点数转为整数还会丢弃小数部分；使用前应明确这是否符合任务要求。

In [8]:
counts = np.array([100, 300], dtype=np.int16)
narrow = counts.astype(np.int8)
measurements = np.array([2.75, 4.25], dtype=np.float64)
whole = measurements.astype(np.int16)

print(narrow, narrow.dtype)  # [100  44] int8，300 超出 int8 范围。
print(counts)  # [100 300]，原数组保留。
print(whole, whole.dtype)  # [2 4] int16，转换不是四舍五入。
print(narrow.nbytes, counts.nbytes)  # 2 4，节省空间伴随范围变化。

[100  44] int8
[100 300]
[2 4] int16
2 4


### 6.2 整数运算溢出

NumPy 整数的位宽固定，运算不会为了容纳结果而自动无限扩展。Python int 则可以随着数值增大扩展。

下面输入 120 和增量 10 都能放进 int8，但它们的和不能。必须在运算之前转为足够宽的类型；溢出后再转换不能找回已经丢失的值。整数数组的此类溢出可能不发出警告。

In [9]:
counts = np.array([120], dtype=np.int8)
overflowed = counts + 10
widened_first = counts.astype(np.int16) + 10
widened_after = overflowed.astype(np.int16)

print(overflowed, overflowed.dtype)  # [-126] int8，发生溢出。
print(widened_first, widened_first.dtype)  # [130] int16，先扩大范围。
print(widened_after)  # [-126]，事后转换没有恢复数学上的结果。
print(120 + 10)  # Python int 得到 130。

[-126] int8
[130] int16
[-126]
130


### 6.3 Python 标量参与类型提升

混合类型运算需要确定共同的结果 dtype，这称为类型提升（type promotion）。NumPy 2 系列会考虑 Python 标量的数值种类，但不按其精度直接扩大 NumPy 类型。

因此，float32 数组加 Python float 可以保持 float32；int16 数组加 Python int 可以保持 int16。整数数组加 Python float 则得到 float64。

In [10]:
decimals = np.array([1.25, 2.5], dtype=np.float32)
integers = np.array([2, 4], dtype=np.int16)

print((decimals + 0.5).dtype)  # float32，没有自动变为 float64。
print((integers + 10).dtype)  # int16。
print(integers + 0.5, (integers + 0.5).dtype)  # [2.5 4.5] float64。
print((decimals + np.float64(0.5)).dtype)  # float64，显式 NumPy 标量保留精度信息。

float32
int16
[2.5 4.5] float64
float64


Python 整数若不能转换到运算所需的 NumPy 整数类型，会触发 OverflowError。它与“输入都可表示、计算结果溢出”是两种不同情况。

下面 1000 本身就不能放入 int8，因此尚未得到加法结果便失败。

In [11]:
small = np.array([1, 2], dtype=np.int8)

# 预期 OverflowError：1000 超出 int8 的可表示范围，加法前的整数转换失败。
print(small + 1000)

OverflowError: Python integer 1000 out of bounds for int8

In [12]:
print(small.astype(np.int16) + 1000)  # [1001 1002]，先转换后计算。

[1001 1002]


## 7 选学：转换限制与共同类型

### 7.1 casting 参数

astype() 的 casting 参数可以限制转换。safe 按类型判断是否允许保值转换；unsafe 是默认规则，允许有损转换。

NumPy 2.4 起还提供 same_value，检查当前元素的值是否改变。它能允许值恰好是整数的浮点数组转为整数，但遇到小数丢失或溢出会触发 ValueError。

In [13]:
integers = np.array([10, 20], dtype=np.int16)
print(integers.astype(np.int32, casting="safe"))  # [10 20]，扩大整数范围。

exact = np.array([2.0, 4.0])
print(exact.astype(np.int16, casting="same_value"))  # [2 4]，当前值不变。

fractional = np.array([2.5, 4.0])

# 预期 ValueError：same_value 不允许把 2.5 转成整数后丢失小数部分。
print(fractional.astype(np.int16, casting="same_value"))

[10 20]
[2 4]


ValueError: could not cast 'same_value' double to short

### 7.2 result_type 与混合整数

np.result_type() 根据输入类型查询共同类型。混合有符号和无符号整数时，结果可能需要更宽的类型；int64 与 uint64 的共同类型甚至是 float64，因为没有更宽的有符号整数类型容纳双方范围。

共同类型不保证每种运算都使用该类型，例如整数真除法仍返回浮点结果。

In [14]:
print(np.result_type(np.int8, np.uint8))  # int16。
print(np.result_type(np.int64, np.uint64))  # float64，不是更宽的整数。

values = np.array([2, 4], dtype=np.int16)
print(np.result_type(values, 2))  # int16，共同类型。
print((values / 2).dtype)  # float64，真除法有自己的结果类型要求。

int16
float64
int16
float64


## 本章小结

（1）dtype 决定元素的表示方式；astype() 默认生成同形的新数组，转换可能改变数值。

（2）itemsize 和 nbytes 描述元素存储量，iinfo 和 finfo 帮助检查范围与浮点精度。

（3）计算前检查目标范围；整数溢出后再加宽类型，不能恢复正确结果。

（4）NumPy 标量与 Python 标量的类型提升规则不同。看到较小 dtype 时，应能解释 Python 数值参与运算后是否仍保留它。

## 练习

（1）用 int16 保存下面四个计数，再转换为 float32。打印两者的 shape、dtype、itemsize 与 nbytes，并解释元素总数不变时字节数为何变化。

In [15]:
counts = [10, 20, 30, 40]

# 在此创建两种类型的数组并打印属性。
# 检查：shape 均为 (4,)；int16 共 8 字节，float32 共 16 字节。
# 再打印原数组，确认 astype() 没有改变其 dtype。

（2）先预测下面三个结果的值与 dtype，再运行核对。说明“先转换后计算”和“先计算后转换”的区别。

In [16]:
counts = np.array([125], dtype=np.int8)

# 先在此记录预测；核对时结合 iinfo(np.int8) 的范围解释。
print(counts + 5)
print((counts + 5).astype(np.int16))
print(counts.astype(np.int16) + 5)
# 补充打印各结果的 dtype。

[-126]
[-126]
[130]


（3）原始记录在 0～200 之间，后续每项最多增加 100。从 uint8、int16 中选择一种计算类型，用 iinfo() 支持你的理由，再计算示例结果。如果仅保存原始记录、完全不计算，你的选择是否会变化？

In [17]:
raw_counts = [0, 100, 200]
increment = 100

# 在注释中说明两种任务的选择理由，再创建数组和计算。
# 检查：结果应为 [100, 200, 300]，不能溢出。
# 比较两种类型的 itemsize，说明取值范围与存储量的取舍。

（4）把下面的浮点读数转为 int16，观察小数变化。若任务要求所有原始小数保持不变，说明是否应进行该转换；选学完成者可以用 same_value 检查这一条件。

In [18]:
readings = np.array([3.0, 3.75, 4.0], dtype=np.float64)

# 在此转换并打印结果，同时打印原数组。
# 检查：第二个值会改变；不要把转换成功等同于数值无损。
# 在注释中说明保持原始小数时的处理选择。

### 重点练习提示

对应第（3）题。先独立完成，再按需要查看提示。

（1）类型应容纳计算后的最大值，而不只是当前记录的最大值。

（2）分别查看两种类型的 iinfo 上界与 itemsize，并把 200 + 100 与上界比较。

### 重点练习参考解析

对应第（3）题。

计算类型选择 int16：uint8 的范围为 0～255，容纳不了 300；int16 的范围为 −32768～32767。先用 int16 创建原始数组，再加 100，结果为 [100, 200, 300]，形状为 (3,)。

只保存 0～200 的原始整数且完全不计算时，可选 uint8，每项 1 字节，三项共 3 字节；int16 每项 2 字节，共 6 字节。先在 uint8 中计算再转 int16，不能恢复溢出前的结果。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| NumPy 官方文档（NumPy 2.5） | 类型：[Data types](https://numpy.org/doc/2.5/user/basics.types.html) 的 Numerical Data Types、Array scalars、Overflow errors；[Data type objects](https://numpy.org/doc/2.5/reference/arrays.dtypes.html) 的 dtype 描述；[Scalars](https://numpy.org/doc/2.5/reference/arrays.scalars.html) 的数值标量与 Sized aliases；[array](https://numpy.org/doc/2.5/reference/generated/numpy.array.html) 的 dtype 推断。存储：[itemsize](https://numpy.org/doc/2.5/reference/generated/numpy.ndarray.itemsize.html) 的字节定义；[nbytes](https://numpy.org/doc/2.5/reference/generated/numpy.ndarray.nbytes.html) 的 Notes 与元素数量关系。范围：[iinfo](https://numpy.org/doc/2.5/reference/generated/numpy.iinfo.html) 的 min、max；[finfo](https://numpy.org/doc/2.5/reference/generated/numpy.finfo.html) 的 min、max、eps。转换：[astype](https://numpy.org/doc/2.5/reference/generated/numpy.ndarray.astype.html) 的 dtype、copy、casting、Raises 与示例，same_value 自 2.4 提供。提升：[Data type promotion](https://numpy.org/doc/2.5/reference/arrays.promotion.html) 的 Detailed behavior of Python scalars、Numerical promotion、Exceptions；[result_type](https://numpy.org/doc/2.5/reference/generated/numpy.result_type.html) 的返回类型与参数。 |